# T3A — Pose / keypoints — camera & studio shift

**Lemma D3** · `nuisance="augmentation"` · [Task doc](../../docs/tasks/t03a-pose-gradient.md) · FINAL: `paper_code/T3/Task3A/FINAL.md`

> E1_aniso subspace PMH on COCO pose: **54.49%** clean PCK@0.05 (+22.4 pp vs baseline 32.07%).

| § | What you do |
|---|-------------|
| 1–4 | Install → load demo → `check_applicability` |
| 5–6 | Estimate $\Sigma_{\text{task}}$ → PMH train → Step 5 on deploy holdout |
| 7–8 | Reproduce paper scripts → plug in your data |


## 1 — Install


In [ ]:
!pip install -q matching-pmh torch


## 2 — Config & imports


In [ ]:
import os
import torch
from pmh.benchmark.presets import get_preset
from pmh.pytorch_eval import (
    pytorch_demo_loaders,
    pytorch_isotropic_demo_loaders,
    pytorch_multilayer_vision_demo_loaders,
    pytorch_sequence_demo_loaders,
)
from pmh import PMHConfig, PMHTrainer, evaluate_robust_fit, check_applicability, suggest_nuisance
from pmh.adoption import RECIPE_ONE_LINER, format_recipe_banner

QUICK = os.environ.get("PMH_QUICK", "").lower() in ("1", "true", "yes")
EPOCHS = 2 if QUICK else 6
SEED = 0
print(RECIPE_ONE_LINER)


## 3 — Load demo data


In [ ]:
preset = None
N = 200 if QUICK else 500
bundle = pytorch_demo_loaders(n=N, batch_size=32, seed=SEED)
model = bundle.model
hook, head = bundle.encoder, bundle.head
train_loader, src_loader, tgt_loader, val_loader = (
    bundle.train_loader, bundle.source_batches, bundle.target_batches, bundle.val_loader,
)

def aug_noise(x):
    return x + 0.12 * torch.randn_like(x)

def aug_scale(x):
    return x * (0.9 + 0.2 * torch.rand(x.size(0), 1, device=x.device, dtype=x.dtype))

augmentations = [aug_noise, aug_scale]


## 4 — Scope (applicability)


In [ ]:
from pmh import check_applicability, suggest_nuisance

print(suggest_nuisance(has_source_labels=True, has_target_domain=True))
app = check_applicability(stack="pytorch", has_target_domain=True)
print(app.summary())
print("suggested nuisance:", app.suggested_nuisance, "(expect 'augmentation')")


## 5 — Estimate $\Sigma_{\text{task}}$ + PMH train


In [ ]:
import copy
from pmh import PMHTrainer, PMHConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
m = copy.deepcopy(model).to(device)
trainer = PMHTrainer(
    m, hook=hook, head=head, nuisance="augmentation", rank=16, pmh_config=PMHConfig.balanced(), device=device,
)
trainer.estimate(source_batches=src_loader, augmentations=augmentations)
trainer.fit(train_loader, source_batches=src_loader, epochs=EPOCHS  # estimate done above)
print("preflight", trainer.artifact_.preflight, "method", getattr(trainer.artifact_, "method", None))


## 6 — Step 5 (deploy holdout)


In [ ]:
from pmh import evaluate_robust_fit

report = evaluate_robust_fit(
    m, train_loader, val_loader,
    source_batches=src_loader, target_batches=tgt_loader,
    hook=m.enc, head=m.head, nuisance="augmentation", rank=16, 
    pmh_config=PMHConfig.balanced(), epochs=max(2, EPOCHS - 2), include_falsification=True, seed=SEED,
)
print(report.summary())
if hasattr(report, "baseline_metric"):
    print("deploy holdout — baseline:", report.baseline_metric, "pmh:", report.pmh_metric)


## 7 — Paper reproduction


Frozen results: `paper_code/T3/Task3A/FINAL.md`

- **Calibrate occlusion subspace W:** `python paper_code/T3/Task3A/calibrate_subspace.py`
- **Train baseline / E1 / E1_aniso / VAT:** `python paper_code/T3/Task3A/train.py`
- **Robustness + embedding eval:** `python paper_code/T3/Task3A/eval.py`


## 8 — Your pipeline


Swap demo loaders for your `train_loader`, `source_batches`, `target_batches`, and deploy holdout. Hook the backbone before your task head.
